In [1]:
%pip install flask
%pip install anthropic python-dotenv
from dotenv import load_dotenv
load_dotenv()
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)
INPUT_PRICE = 1.00       # دولار لكل مليون توكن مُدخل (Haiku 4.5)
OUTPUT_PRICE = 5.00      # دولار لكل مليون توكن مُخرج (Haiku 4.5)
CACHE_READ_PRICE = 0.10  # 10% من سعر المُدخل
CACHE_WRITE_PRICE = 1.25 # 125% من سعر المُدخل (كاش 5 دقايق)


def chat(messages, system=None):
    system_content = system if system else system_prompt
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        system=[{
            "type": "text",
            "text": system_content,
            "cache_control": {"type": "ephemeral"}
        }],
        messages=messages
    )

    # الأرقام الحقيقية من الرد نفسه
    u = message.usage
    cost = (
        (u.input_tokens / 1_000_000) * INPUT_PRICE +
        (u.output_tokens / 1_000_000) * OUTPUT_PRICE +
        (u.cache_read_input_tokens / 1_000_000) * CACHE_READ_PRICE +
        (u.cache_creation_input_tokens / 1_000_000) * CACHE_WRITE_PRICE
    )

    print(f"[توكن مُدخل: {u.input_tokens} | كاش مقروء: {u.cache_read_input_tokens} | "
          f"كاش مكتوب: {u.cache_creation_input_tokens} | مُخرج: {u.output_tokens} | "
          f"تكلفة هذا الطلب: ${cost:.5f}]")

    return message.content[0].text


In [3]:
import json
knowledge_base= """

كيف أضيف تصنيف للأصناف؟
تصنيف الأصناف يقسّم الأصناف إلى مجموعات لتسهيل الوصول إليها. من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة تصنيف الأصناف. اضغط "جديد"، أدخل اسم التصنيف، ثم اضغط حفظ.
كيف أدخل للنظام؟
أدخل من خلال أيقونة سطح المكتب، أو من خلال رابط في حالة كان سحابي.
كيف أضيف صنف؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة بيانات الأصناف. اضغط "جديد" وأدخل بيانات الصنف الأساسية (التصنيف الرئيسي، اسم الصنف، نوع الصنف، الوحدة الرئيسية، التكلفة الأولية) ثم اضغط حفظ. بعدها: أدخل سعر البيع من تبويب "سعر البيع"، اقرأ الباركود الدولي من تبويب "الباركود".
كيف أطبع باركود محلي؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة بيانات الأصناف. من تبويب بيانات الصنف الضغط على زر بحث متقدم، كتابة اسم الصنف والضغط على زر بحث ثم الضغط على رقم الصنف لاختياره، ثم اطبع الباركود المحلي من تبويب "طباعة الباركود".
كيف أضيف وحده للصنف؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة بيانات الأصناف. من خانة "وحدات الصنف"، أضف الوحدات الإضافية للصنف إن وجدت، ثم اضغط حفظ.
كيف أدخل رصيد افتتاحي لصنف؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة الأرصدة الافتتاحية. اختر التصنيف الرئيسي واضغط بحث، سيظهر جدول بالأسفل يضم رقم الصنف واسمه. أدخل الكمية المتوفرة في المخزون والتكلفة الأولية، ثم اضغط حفظ ثم ترحيل.
كيف أضيف صورة للصنف؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة بيانات الأصناف. من تبويب بيانات الصنف الضغط على زر بحث متقدم، كتابة اسم الصنف والضغط على زر بحث ثم الضغط على رقم الصنف لاختياره، ثم إضافة صورة من تبويب
"صورة الصنف".
كيف أضيف فاتورة المبيعات؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة فاتورة مبيعات. اضغط "جديد"، أدخل رقم الصنف أو اكتب اسم الصنف واختره ليُضاف إلى الفاتورة، وستظهر الوحدة والكمية والسعر تلقائيًا مع إمكانية التعديل. يمكن أيضًا تغيير العملة والفرع والصندوق والمخزن وطريقة الدفع وتاريخ الفاتورة والعميل، ثم اضغط حفظ.
كيف أضيف فاتورة المشتريات؟
من الشاشة الرئيسية، افتح نظام المشتريات، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة فاتورة المشتريات. اضغط "جديد"، أدخل رقم الصنف أو اكتب اسم الصنف واختره ليُضاف إلى الفاتورة، وستظهر الوحدة والكمية والسعر تلقائيًا مع إمكانية التعديل. يمكن أيضًا تغيير العملة والفرع والصندوق والمخزن وطريقة الدفع وتاريخ الفاتورة والمورد، ثم اضغط حفظ.
كيف أضيف فاتورة مردود المشتريات؟
من الشاشة الرئيسية، افتح نظام المشتريات، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة فاتورة مردود مشتريات. اضغط "جديد"، ثم ابحث عن الفاتورة الأصلية برقمها (بحث متقدم)، حدد المورد وطريقة الدفع والكمية المراد إرجاعها، ثم اضغط حفظ.
كيف أضيف فاتورة مردود المبيعات؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة مردود المبيعات. اضغط "جديد"، ثم ابحث عن الفاتورة الأصلية برقمها (بحث متقدم)، حدد العميل وطريقة الدفع والكمية المراد إرجاعها، ثم اضغط حفظ.
كيف أفعل سند قبض؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة سندات القبض. اضغط "جديد"، اختر نوع الوثيقة وأدخل البيان ثم أدخل الحساب والمبلغ والبيان، اضغط على زر أضف ثم اضغط حفظ لعرض التقرير.
كيف أفعل سند صرف؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة سندات الصرف. اضغط "جديد"، اختر نوع الوثيقة وأدخل البيان، ثم أدخل الحساب والمبلغ والبيان، اضغط على زر أضف ثم اضغط حفظ لعرض التقرير.
كيف أفعل أمر توريد؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة أمر التوريد. اضغط "جديد"، أضف الصنف بالضغط على المربع الأخضر ثم كتابة اسم الصنف والبحث عنه واختيار صنف أو أكثر ثم الضغط على زر موافق، وستظهر الوحدة والكمية والسعر تلقائيًا مع إمكانية التعديل، ويمكن تغيير الفرع والمخزن وإضافة الحساب، ثم اضغط حفظ لعرض التقرير.
كيف أفعل أمر صرف؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة أمر صرف مخزني. اضغط "جديد"، أضف الصنف بالضغط على المربع الأخضر ثم كتابة اسم الصنف والبحث عنه واختيار صنف أو أكثر ثم الضغط على زر موافق، وستظهر الوحدة والكمية والسعر تلقائيًا مع إمكانية التعديل، ويمكن تغيير الفرع والمخزن وإضافة الحساب، ثم اضغط حفظ لعرض التقرير.
كيف أطلع كشف حساب لعميل؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة كشف حساب خلال فترة. حدد السنة المالية والفترة المحاسبية والجهة (عميل) ونوع التقرير ورقمه، وحدد الفترة (من - إلى)، ثم اضغط عرض التقرير.
كيف أطلع كشف حساب لمورد؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة كشف حساب خلال فترة. حدد السنة المالية والفترة المحاسبية والجهة (مورد) ونوع التقرير ورقمه، وحدد الفترة (من - إلى)، ثم اضغط عرض التقرير.
كيف أضيف صندوق؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم تهيئة الأستاذ العام من شريط الأقسام الجانبي، واختر شاشة الصناديق. اضغط "جديد"، أدخل اسم الصندوق ورقم الحساب والفرع، ثم اضغط حفظ.
كيف أضيف مخزن؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة المخازن. اضغط "جديد"، أدخل اسم المخزن وحدد إن كان قابلًا للبيع والفرع، ثم اضغط حفظ.
كيف أفعل إعدادات للحسابات؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم تهيئة الأستاذ العام من شريط الأقسام الجانبي، واختر شاشة إعدادات الحسابات. حدد الفرع والنوع والخيارات المطلوبة، ثم اضغط حفظ.
كيف أضيف عملة جديدة؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم تهيئة الأستاذ العام من شريط الأقسام الجانبي، واختر شاشة تهيئة العملات. اضغط "جديد"، أدخل اسم العملة ورمزها وسعر التحويل، ثم اضغط حفظ.
كيف أفعل قيود يومية؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة قيود اليومية. اضغط "جديد"، أدخل الحساب والبيان ومبلغ المدين أو الدائن ثم اضغط "أضف"، وكرر لبقية البنود، ثم اضغط حفظ لعرض تقرير القيد.
كيف استعرض فواتير مبيعات السابقة؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة استعراض العمليات. حدد الفرع ونوع الوثيقة وطريقة الدفع والفترة، ثم اضغط بحث. اضغط على رقم الوثيقة للانتقال إليها، أو على السهم المجاور لاستعراضها في نفس الصفحة.
كيف استعرض فواتير مشتريات السابقة؟
من الشاشة الرئيسية، افتح نظام المشتريات، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة استعراض العمليات. حدد الفرع ونوع الوثيقة وطريقة الدفع والفترة، ثم اضغط بحث. اضغط على رقم الوثيقة للانتقال إليها، أو على السهم المجاور لاستعراضها في نفس الصفحة.
كيف أظهر تقرير المبيعات اليومية؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير المبيعات اليومية. حدد الفرع والمخزن وطريقة الدفع والفترة، ثم اضغط بحث لعرض التقرير.
كيف أظهر تقرير المبيعات الإجمالية؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير إجمالي بالمبيعات. حدد الفرع والمخزن وطريقة الدفع والفترة، ثم اضغط بحث لعرض التقرير.
كيف أظهر تقرير مبيعات التفصيلية؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير تفصيلي بالمبيعات. حدد الفرع والمخزن وطريقة الدفع والفترة، ثم اضغط بحث لعرض التقرير.
كيف أظهر تقرير المشتريات الإجمالية؟
من الشاشة الرئيسية، افتح نظام المشتريات، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير إجمالي بالمشتريات. حدد الفرع والمخزن وطريقة الدفع والفترة، ثم اضغط بحث لعرض التقرير.
كيف أظهر تقرير المشتريات التفصيلية؟
من الشاشة الرئيسية، افتح نظام المشتريات، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير تفصيلي بالمشتريات. حدد الفرع والمخزن وطريقة الدفع والفترة، ثم اضغط بحث لعرض التقرير.
كيف أظهر تقرير سندات الصرف الإجمالية؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير إجمالي بسندات الصرف. حدد الفرع ونوع الوثيقة والسنة المالية والفترة، ثم اضغط بحث لعرض التقرير.
كيف أظهر تقرير سندات القبض الإجمالية؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير إجمالي بسندات القبض. حدد الفرع ونوع الوثيقة والسنة المالية والفترة، ثم اضغط بحث لعرض التقرير.
كيف أغير الفرع والسنة المالية؟
من أي نظام، اضغط على اسم المستخدم أو الصورة أعلى يسار الشاشة، واختر "تغيير الفرع والسنة المالية" من القائمة. اضغط "عرض بيانات"، حدد السنة المالية والفرع المطلوب، ثم اضغط حفظ.
كيف أعدل الفترة المحاسبية؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم تهيئة الأستاذ العام من شريط الأقسام الجانبي، واختر شاشة الفترات المحاسبية. اضغط "جديد"، حدد السنة المالية وبداية الفترة ونهايتها، ثم اضغط حفظ.
كيف أضيف بنك؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم تهيئة الأستاذ العام من شريط الأقسام الجانبي، واختر شاشة البنوك. اضغط "جديد"، أدخل اسم البنك ورقم الحساب، ثم اضغط حفظ.
كيف أعدل إعدادات الحسابات؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم تهيئة الأستاذ العام من شريط الأقسام الجانبي، واختر شاشة إعدادات الحسابات. حدد الفرع، ثم النوع (عرض أو نموذج)، ثم الخيار المناسب (السماح بالعرض أو نوع النموذج)، ثم اضغط حفظ.
كيف أضيف فرع؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة الفروع. اضغط "جديد"، أدخل اسم الفرع والرقم الضريبي والعنوان وبقية البيانات، ثم اضغط حفظ.
كيف أضيف الدليل المحاسبي؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة الدليل المحاسبي. اضغط "جديد"، أدخل رقم الحساب واسمه والحساب الرئيسي، ثم اضغط حفظ.
كيف أضيف مركز تكلفة؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة مراكز التكلفة. اضغط "جديد"، أدخل اسم مركز التكلفة ونوع التصنيف ورقمه، ثم اضغط حفظ.
كيف أفعل ترحيل للمستندات؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم الترحيل من شريط الأقسام الجانبي، واختر شاشة الترحيل. حدد الفرع ونوع الوثيقة والفترة والحالة، ثم اضغط بحث سيظهر جدول الترحيل، الضغط على المربع في آخر عمود في اليسار للوثيقة المراد ترحيلها ثم الضغط على زر ترحيل.
كيف أرحل الفواتير إلى الزكاة والدخل؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم الترحيل من شريط الأقسام الجانبي، واختر شاشة ترحيل الفواتير إلى الزكاة. حدد الفرع ونوع الوثيقة والفترة، ثم اضغط بحث لعرض التقرير.
كيف أفعل قوائم مالية؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم القوائم المالية من شريط الأقسام الجانبي، واختر شاشة القوائم المالية. حدد الفرع ونوع التقرير والفترة، ثم اضغط عرض التقرير.
كيف أفعل ميزان مراجعة؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم ميزان المراجعة من شريط الأقسام الجانبي، واختر شاشة ميزان المراجعة. حدد الفرع ونوع التقرير والفترة، ثم اضغط بحث لعرض الجدول.
كيف أفعل تقرير قيود يومية؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقارير قيود اليومية. حدد الفرع ونوع الوثيقة والفترة، ثم اضغط عرض التقرير.
كيف أفعل تقرير سندات القبض التفصيلية؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير تفصيلي بسندات القبض. حدد الفرع ونوع الوثيقة والسنة المالية والفترة، ثم اضغط بحث لعرض التقرير.
كيف أفعل تقرير سندات الصرف التفصيلية؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير تفصيلي بسندات الصرف. حدد الفرع ونوع الوثيقة والسنة المالية والفترة، ثم اضغط بحث لعرض التقرير.
كيف أفعل تقرير الإقرار الضريبي؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير الإقرار الضريبي. حدد الفرع ونوع الفاتورة والسنة المالية والفترة، ثم اضغط بحث لعرض التقرير.
كيف أفعل تقرير الفواتير المرحلة إلى الزكاة؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة الفواتير المرحلة إلى الزكاة. حدد الفرع ونوع الوثيقة والفترة، ثم اضغط بحث لعرض التقرير.
كيف أضيف مستخدم جديد؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم المستخدمين والصلاحيات من شريط الأقسام الجانبي، واختر شاشة المستخدمين والصلاحيات. اضغط "جديد"، أدخل اسم المستخدم والاسم الرباعي ونوع المستخدم، ثم اضغط حفظ.
كيف أتحكم بصلاحيات المستخدم؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم المستخدمين والصلاحيات من شريط الأقسام الجانبي، واختر شاشة المستخدمين والصلاحيات. من قائمة المستخدمين أسفل الشاشة، اضغط تعديل (المربع الأزرق) بجانب المستخدم، ثم عدّل الصلاحيات في الجدول أعلاه، وبيانات المستخدم إن لزم، ثم اضغط حفظ.
كيف أفعل نسخة احتياطية؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم المستخدمين والصلاحيات من شريط الأقسام الجانبي، واختر شاشة نسخة احتياطية. من جدول نسخ قاعدة البيانات، اضغط تنزيل، ثم اضغط زر نسخ.
كيف أراقب حركة المستخدمين؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم المستخدمين والصلاحيات من شريط الأقسام الجانبي، واختر شاشة مراقبة حركة المستخدمين. حدد الفرع والنظام والمستخدم، ثم اضغط بحث لعرض حركة المستخدمين.
كيف أعدل ترحيل العمليات الحسابية؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم تهيئة الأستاذ العام من شريط الأقسام الجانبي، واختر شاشة الإعدادات العامة. حدد طريقة ترحيل العمليات المحاسبية (تلقائي أو بعد المراجعة)، ثم اضغط حفظ.
كيف أعدل صنف؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة بيانات الأصناف. اضغط بحث متقدم، اكتب اسم الصنف واضغط بحث، اختر الصنف واضغط موافق. اضغط تعديل، عدّل البيانات المطلوبة، ثم اضغط حفظ.
كيف أعدل على فاتورة المشتريات؟
من الشاشة الرئيسية، افتح نظام المشتريات، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة فاتورة المشتريات. أدخل رقم الفاتورة أعلى اليمين واضغط بحث. اضغط تعديل أسفل الفاتورة، عدّل البيانات المطلوبة (اسم الصنف، رقم فاتورة المورد، الوحدة، إلخ)، ثم اضغط حفظ.
كيف يتم تعليق الترحيل لفاتورة المشتريات؟
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم الترحيل من شريط الأقسام الجانبي، واختر شاشة الترحيل. اختر الحالة "مرحّل"، ونوع الوثيقة "فاتورة مشتريات"، والتاريخ، ثم اضغط بحث. في الجدول أسفل الشاشة، اضغط المربع في آخر عمود يسار حتى تظهر علامة صح، ثم اضغط "إلغاء الترحيل".
كيف أضيف وحدة قياس؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم تهيئة المستودعات من شريط الأقسام الجانبي، واختر شاشة وحدات القياس. اضغط "جديد"، أدخل اسم الوحدة، وحدد المربع بجانبها لجعلها وحدة رئيسية إن لزم، ثم اضغط حفظ.
كيف استرد الأصناف؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم تهيئة المستودعات من شريط الأقسام الجانبي، واختر شاشة استيراد الأصناف. حدد الفرع والمخزن، ارفع الملف من خانة "ارفع ملف"، ثم اضغط "ارفع".
كيف أطلع تقرير حركة الصنف الإجمالي؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير حركة الصنف إجمالي. حدد الفرع والسنة المالية والفترة (من - إلى)، ثم اضغط بحث لعرض التقرير.
كيف أطلع تقرير حركة الصنف التفصيلي؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير حركة الصنف تفصيلي. حدد الفرع والسنة المالية والفترة (من - إلى)، ثم اضغط بحث لعرض التقرير.
كيف أطلع تقرير بأسعار البيع؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير أسعار البيع. من تبويب البحث عن صنف، اضغط السهم الأزرق لفتح البحث المتقدم، أدخل اسم الصنف واضغط بحث، اختر الصنف واضغط موافق، ثم اضغط بحث لعرض التقرير.
كيف أطبع باركود صنف؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة طباعة باركود الأصناف. اختر التصنيف الرئيسي واسم الصنف، ثم اضغط بحث. حدد مربع الصنف الذي يظهر أسفل الشاشة، ثم اضغط طباعة.
كيف أفعل تقرير إجمالي بأوامر الصرف؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير إجمالي بأوامر الصرف. حدد الفرع والسنة المالية والفترة ونوع الأمر، ثم اضغط بحث لعرض التقرير.
كيف أفعل تقرير تفصيلي بأوامر الصرف؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير تفصيلي بأوامر الصرف. حدد الفرع والسنة المالية والفترة ونوع الأمر، ثم اضغط بحث لعرض التقرير.
كيف أفعل تقرير إجمالي بأوامر التوريد؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير إجمالي بأوامر التوريد. حدد الفرع والسنة المالية والفترة ونوع الأمر، ثم اضغط بحث لعرض التقرير.
كيف أفعل تقرير تفصيلي بأوامر التوريد؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير تفصيلي بأوامر التوريد. حدد الفرع والسنة المالية والفترة ونوع الأمر، ثم اضغط بحث لعرض التقرير.
كيف أفعل تقرير تفصيلي بالتحويلات المخزنية؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير تفصيلي بالتحويلات المخزنية. حدد الفرع والسنة المالية والفترة، ثم اضغط بحث لعرض التقرير.
كيف أفعل تقرير إجمالي بالتحويلات المخزنية؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير إجمالي بالتحويلات المخزنية. حدد الفرع والسنة المالية والفترة، ثم اضغط بحث لعرض التقرير.
كيف أفعل تقرير إجمالي بالاستلام المخزني؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير إجمالي بالاستلامات المخزنية. حدد الفرع والسنة المالية والفترة، ثم اضغط بحث لعرض التقرير.
كيف أفعل تقرير تفصيلي بالاستلام المخزني؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير تفصيلي بالاستلامات المخزنية. حدد الفرع والسنة المالية والفترة، ثم اضغط بحث لعرض التقرير.
كيف أفعل تقرير مراقبة مخزن؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير مراقبة المخزون. حدد الفرع والمخزن والتصنيف الرئيسي، ثم اضغط بحث لعرض التقرير.
كيف أدخل الجرد اليومي؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة إدخال الجرد اليدوي. اضغط "جديد"، حدد الفرع والمخزن وتاريخ الجرد. أضف الصنف بالضغط على السهم الأخضر لفتح البحث المتقدم، اكتب اسم الصنف واضغط موافق لتظهر معلومات الصنف.
كيف أفعل تحويل مخزني؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة تحويل مخزني. حدد المخزن المرسل والمستقبل وتاريخ التحويل. أضف الصنف بالضغط على السهم الأخضر لفتح البحث المتقدم، اكتب اسم الصنف واضغط موافق، ثم اضغط حفظ لعرض تقرير التحويل.
كيف أذهب إلى الإعدادت العامة في المخازن؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم تهيئة المستودعات من شريط الأقسام الجانبي، واختر شاشة إعدادات المخازن. حدد الفرع ثم النوع (نموذج طباعة الباركود، نموذج التحويل المخزني، إلخ)، اختر النموذج من القائمة، ثم اضغط حفظ.
كيف أعدل إعدادات (متغيرات) المخازن؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم تهيئة المستودعات من شريط الأقسام الجانبي، واختر شاشة متغيرات المخازن. عدّل الإعدادات المطلوبة (استخدام تاريخ الانتهاء، نوع الجرد، طريقة حساب التكاليف)، ثم اضغط حفظ.
كيف أسعر الصنف؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة تسعير الأصناف. حدد الفرع ومستوى التسعيرة والتصنيف الرئيسي واسم الصنف، عدّل السعر من تبويب "سعر البيع"، ثم اضغط حفظ.
كيف أفعل الاستلام المخزني؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة استلام مخزني. حدد المخزن والحالة وتاريخ التحويل، اكتب رقم التحويل واضغط بحث، ثم اضغط حفظ لعرض تقرير الاستلام.
كيف أفعل أمر التجميع؟
من الشاشة الرئيسية، افتح نظام المخازن، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة أمر التجميع. حدد المخزن المرسل والمستقبل وتاريخ الأمر وقيمته. أضف الصنف بالضغط على السهم الأخضر لفتح البحث المتقدم، اكتب اسم الصنف واضغط موافق، ثم اضغط حفظ.
كيف أذهب إلى إعدادات المشتريات؟
من الشاشة الرئيسية، افتح نظام المشتريات، ثم قسم تهيئة المشتريات من شريط الأقسام الجانبي، واختر شاشة إعدادات المشتريات. حدد الفرع ثم النوع. يمكن تغيير طريقة إرسال الفاتورة، نموذج فاتورة المشتريات، نموذج طلب الشراء، وإظهار الكمية المجانية أو بالوحدة، ثم اضغط حفظ.
كيف أضيف نوع المورد؟
من الشاشة الرئيسية، افتح نظام المشتريات، ثم قسم تهيئة المشتريات من شريط الأقسام الجانبي، واختر شاشة أنواع الموردين. اضغط "جديد"، اكتب نوع المورد، ثم اضغط حفظ.
كيف أضيف بيانات المورد؟
من الشاشة الرئيسية، افتح نظام المشتريات، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة بيانات الموردين. اضغط "جديد"، أدخل اسم المورد ونوعه وفئته وبقية البيانات، ثم اضغط حفظ.
كيف أفعل تقرير تفصيلي بمشتريات صنف؟
من الشاشة الرئيسية، افتح نظام المشتريات، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير تفصيلي بمشتريات صنف. حدد الفرع والمخزن وطريقة الدفع والفترة. من تبويب البحث عن صنف، اضغط السهم الأزرق، أدخل اسم الصنف واضغط بحث، ثم اختر رقم الصنف لعرض التقرير.
كيف أضيف إعدادات للمركبات؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم تهيئة المبيعات من شريط الأقسام الجانبي، واختر شاشة إعدادات المركبات. من تبويب بيانات الماركة، اضغط "جديد"، أدخل اسم الماركة، ثم اضغط حفظ.
كيف أضيف شروط عقود المبيعات؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم تهيئة المبيعات من شريط الأقسام الجانبي، واختر شاشة شروط عقد مبيعات. اضغط "جديد"، اختر نوع العقد، ثم من قائمة الشرط الرئيسي اضغط "أضف" واكتب الشرط ورقم الترتيب، ثم اضغط حفظ.
كيف أضيف نوع عميل جديد؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم تهيئة المبيعات من شريط الأقسام الجانبي، واختر شاشة أنواع العملاء. اضغط "جديد"، اكتب نوع العميل، ثم اضغط حفظ.
كيف أعرف الورديات؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم تهيئة المبيعات من شريط الأقسام الجانبي، واختر شاشة تعريف الورديات. اضغط "جديد"، اكتب اسم الوردية وحدد وقت البداية والنهاية، ثم اضغط حفظ.
كيف أضيف عميل جديد؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة بيانات العملاء. اضغط "جديد"، أدخل اسم العميل ونوعه وفئته وبقية البيانات، ثم اضغط حفظ.
كيف أضيف عمولات الموظفين؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة عمولات الموظفين. اختر نوع الموظف واسمه من القائمة، اضغط تعديل، عدّل نسبة العمولة، ثم اضغط حفظ.
كيف أضيف موظف؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة الموظفين. اضغط "جديد"، أدخل اسم الموظف ونوعه ورقم الحساب وبقية البيانات، ثم اضغط حفظ.
كيف أربط المستخدم بطريقة الدفع؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة إعدادات المستخدمين. من قائمة "ربط المستخدم بطريقة الدفع"، اختر اسم المستخدم وطريقة الدفع، ثم اضغط حفظ.
كيف أربط المستخدم بالبائع؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة إعدادات المستخدمين. من قائمة "ربط المستخدم بالبائع"، اختر اسم المستخدم واسم البائع، ثم اضغط حفظ.
كيف أربط المستخدمين بالمخازن؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة إعدادات المستخدمين. من قائمة "ربط المستخدمين بالمخازن"، اختر اسم المستخدم، ثم من الجدول أسفل الشاشة حدد المخزن بعلامة الصح، ثم اضغط حفظ.
كيف أربط المندوبين بالعملاء؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم المدخلات من شريط الأقسام الجانبي، واختر شاشة ربط المندوبين بالعملاء. اختر العميل، ثم من الجدول الذي يظهر أسفل الشاشة حدد المربع بعلامة الصح، اختر اسم المندوب، ثم اضغط حفظ.
كيف أضيف نقطة مبيعات؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة نقطة مبيعات. اضغط "جديد"، أدخل رقم الصنف أو اكتب اسم الصنف واختره ليُضاف إلى الفاتورة، وستظهر الوحدة والكمية والسعر تلقائيًا مع إمكانية التعديل. يمكن أيضًا تغيير العملة والفرع والصندوق والمخزن وطريقة الدفع وتاريخ الفاتورة وإضافة عميل، ثم اضغط حفظ.
كيف أغلق كاشير؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم العمليات من شريط الأقسام الجانبي، واختر شاشة إغلاق الكاشير. اختر الوردية واسم المستخدم، ثم من جدول الورديات أسفل الشاشة اضغط اختيار، ثم اضغط عرض التقرير.
كيف أطلع تقرير الخصم على مستوى الصنف؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة الخصم على مستوى الصنف. حدد الحالة وتاريخ بداية الخصم ونهايته، ثم اضغط بحث لعرض التقرير.
كيف أفعل تقرير حركة المبيعات؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير حركة المبيعات. حدد الفرع والفترة، ثم اضغط بحث لعرض التقرير.
كيف أفعل تقرير أرباح المبيعات؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير أرباح المبيعات تحليلي. حدد الفرع والسنة المالية والفترة، ثم اضغط بحث لعرض التقرير.
كيف أفعل تقرير بعرض السعر؟
من الشاشة الرئيسية، افتح نظام المبيعات، ثم قسم التقارير من شريط الأقسام الجانبي، واختر شاشة تقرير إجمالي بعروض السعر. حدد الفرع والفترة، ثم اضغط بحث لعرض التقرير.
ما هو رقم خدمة العملاء؟
تفضل هذا رقم خدمة العملاء: +966 55 053 8747 أو +966 50 534 7079

"""

In [4]:
system_prompt=f"""
أنت مساعد ذكي لشركة سمارتكس للأنظمة المحاسبية، رحب بالعميل وقوله أنك مساعد ذكي وكيف أقدر أخدمك فقط.
 مهمتك الإجابة على أسئلة العملاء بالاعتماد فقط على المعلومات التالية:
 <instructions>
1. الاعتماد المطلق على المصدر: أجب فقط بناءً على المعلومات داخل وسم <knowledge_base>. لا تخترع أي خطوات أو إعدادات غير مذكورة.
2. التعامل مع الاستفسارات الخارجية: إذا كان السؤال غير موجود في قاعدة المعرفة، اعتذر بأسلوب لبق وأبلغ العميل بتحويله لفريق الدعم الفني وهذه أرقامهم: +966 55 053 8747 أو +966 50 534 7079.
3. التنسيق والأسلوب:
-أهم شي خلي كل خطوة في سطر لحالها يعني الخطوة الأولى في السطر والثانية في سطر وهكذا وخلي بينهم مسافة.
   - استخدم اللغة العربية الفصحى بلغة بسيطة ومهنية.
   - قم بتنسيق الخطوات الإجرائية على شكل أرقام عربية هندية مرتبة لتسهيل القراءة.
4. إذا كان السؤال غير واضح مثل إضافة فاتورة أو كيف أدخل الإعدادات فأطلب توضيح من العميل بشكل مؤدب، أساله عن النظام المراد استخدامه أو القسم المراد أو الشاشة المراد على حسب نوع السؤال.
5.رحب في العميل فقط في أول رسالة فقط يعني أول مرة يسألك، بعدين إذا سئل سؤال آخر أو طلب عطيه الجواب مباشرة فقط الجواب لا ترحب فيه ثاني.
</instructions>

<knowledge_base>
{knowledge_base}
</knowledge_base>

<examples>
<example>
<user_query>كيف أضيف عملة جديدة؟</user_query>
<assistant_response>
من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم تهيئة الأستاذ العام من شريط الأقسام الجانبي، واختر شاشة تهيئة العملات. اضغط "جديد"، أدخل اسم العملة ورمزها وسعر التحويل، ثم اضغط حفظ.
</assistant_response>
</example>

<example>
<user_query>كيف أربط النظام مع هيئة الزكاة والدخل؟</user_query>
<assistant_response>
عذراً منك، هذه المعلومة غير متوفرة في دليلي الحالي. يرجى الاتصال أو التواصل واتس اب على الدعم الفني عن طريق الرقم التالي: +966 55 053 8747 أو +966 50 534 7079 .
</assistant_response>
</example>

<example>
<user_query>كيف أضيف فاتورة؟</user_query>
<assistant_response>
رجاءً  هل ممكن تحدد لي نوع الفاتورة؟ فاتورة مبيعات أو فاتورة مشتريات؟
</assistant_response>
</example>
</examples>
    """

In [5]:
messages=[]
while True:
    user_input=input(">")
    add_user_message(messages,user_input)
    answer = chat(messages)
    add_assistant_message(messages,answer)
    print(answer)

KeyboardInterrupt: Interrupted by user

In [ ]:
with open("smartext.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

In [ ]:
import json
import re
from statistics import mean

def grade_by_model(test_case, output):
    eval_system_prompt = """
أنت مقيّم جودة صارم ودقيق (Judge) لنظام خدمة عملاء.
مهمتك المباشرة: تقييم مدى تطابق الإجابة الفعلية للشات بوت مع الإجابة النموذجية المعتمدة.

<instructions>
1. اقرأ <question> و <expected_answer> بعناية لفهم المطلوب والخطوات الصحيحة.
2. اقرأ <actual_response> (إجابة الشات بوت الفعلي).
3. قارن الخطوات والمعلومات. تجاهل الاختلافات اللفظية التي لا تغير المعنى العملي.
4. قيّم بناءً على معايير التقييم الموجودة في <scoring_rubric>.
5. اكتب سبب تقييمك أولاً داخل وسم <reasoning>.
6. أخرج النتيجة النهائية بصيغة JSON حصراً داخل وسم <json_output>.
</instructions>

<scoring_rubric>
- [10]: الإجابة تطابق النموذجية تماماً وتتضمن كافة الخطوات الصحيحة.
- [7-9]: الإجابة صحيحة وعملية، لكن ينقصها تفصيل صغير جداً لا يمنع المستخدم من إتمام المهمة.
- [4-6]: الإجابة صحيحة جزئياً، لكن ينقصها خطوة جوهرية، أو لم تذكر الشاشة/النظام الصحيح.
- [1-3]: الإجابة خاطئة، أو مضللة، أو تحتوي على معلومات غير موجودة في النموذج (هلوسة).
</scoring_rubric>

<examples>
<example>
<input>
<question>كيف أضيف صندوق؟</question>
<expected_answer>من الشاشة الرئيسية، افتح نظام الحسابات العامة، ثم قسم تهيئة الأستاذ العام، واختر شاشة الصناديق. اضغط "جديد"، أدخل اسم الصندوق ورقم الحساب والفرع، ثم اضغط حفظ.</expected_answer>
<actual_response>لإضافة صندوق، اذهب إلى نظام الحسابات العامة > تهيئة الأستاذ العام > الصناديق. اضغط جديد، ادخل الاسم ورقم الحساب والفرع واضغط حفظ.</actual_response>
</input>
<output>
<reasoning>الإجابة الفعلية تضمنت كافة الخطوات المطلوبة للوصول للشاشة (نظام الحسابات، تهيئة الأستاذ، الصناديق) وذكرت كل الحقول المطلوبة (الاسم، رقم الحساب، الفرع) مع حفظ. لا توجد معلومات مفقودة أو مهلوسة.</reasoning>
<json_output>
{
  "correct": true,
  "score": 10
}
</json_output>
</output>
</example>

<example>
<input>
<question>كيف أضيف فاتورة مبيعات؟</question>
<expected_answer>من الشاشة الرئيسية، افتح نظام المبيعات، شاشة فاتورة مبيعات. اضغط جديد، أدخل رقم الصنف وسيظهر السعر والكمية. يمكن تغيير العملة، ثم اضغط حفظ.</expected_answer>
<actual_response>افتح شاشة فاتورة المبيعات، اضغط جديد وادخل الصنف، ثم احفظ الفاتورة.</actual_response>
</input>
<output>
<reasoning>الإجابة الفعلية لم تذكر مسار الوصول الصحيح (نظام المبيعات > العمليات) وتجاهلت المعلومات الإضافية الهامة مثل ظهور السعر تلقائياً وإمكانية تغيير طريقة الدفع والفرع. الإجابة غير كافية للمستخدم.</reasoning>
<json_output>
{
  "correct": false,
  "score": 4
}
</json_output>
</output>
</example>
</examples>
"""

    eval_user_prompt = f"""
قم بتقييم الإجابة التالية:

<question>
{test_case["question"]}
</question>

<expected_answer>
{test_case["expected_answer"]}
</expected_answer>

<actual_response>
{output}
</actual_response>
"""

    grading_messages = []
    add_user_message(grading_messages, eval_user_prompt)

    # استدعاء النموذج للتقييم (تأكد من تمرير eval_system_prompt كنظام)
    result = chat(grading_messages, system=eval_system_prompt)

    # ---------------------------------------------------------
    # استخراج البيانات بصرامة وأمان باستخدام تقنية البحث عن الوسوم
    # ---------------------------------------------------------
    reasoning_match = re.search(r'<reasoning>(.*?)</reasoning>', result, re.DOTALL)
    json_match = re.search(r'<json_output>(.*?)</json_output>', result, re.DOTALL)

    reasoning = reasoning_match.group(1).strip() if reasoning_match else "لم يقدم النموذج سبباً."

    if json_match:
        try:
            parsed_json = json.loads(json_match.group(1).strip())
            return {
                "correct": parsed_json.get("correct", False),
                "score": parsed_json.get("score", 1),
                "reasoning": reasoning
            }
        except json.JSONDecodeError:
            pass

    # في حال فشل النموذج بشكل غير متوقع في توليد JSON صحيح
    return {"correct": False, "score": 1, "reasoning": "فشل استخراج التقييم: صيغة مخرجات غير صالحة."}

In [ ]:
def run_test_case(test_case):
    test_messages = []
    add_user_message(test_messages, test_case["question"])

    # هنا نستخدم الـ Chatbot الأساسي للإجابة على السؤال
    output = chat(test_messages)

    # هنا نقوم بتقييم الإجابة
    grade = grade_by_model(test_case, output)

    return {
        "question": test_case["question"],
        "output": output,
        "score": grade["score"],
        "correct": grade["correct"],
        "reasoning": grade["reasoning"],
    }


def run_eval(dataset):
    results = []
    print("بدء عملية التقييم...\n" + "="*50)
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

        status = "✅ ناجح" if result["correct"] else "❌ مخفق"
        print(f"السؤال: {result['question']}")
        print(f"النتيجة: {status} | الدرجة: {result['score']}/10")
        print(f"تعليل المقيّم: {result['reasoning']}")
        print("-" * 50)

    avg = mean([r["score"] for r in results])
    failed = [r for r in results if not r["correct"]]
    print(f"\n📊 التقرير النهائي:")
    print(f"متوسط الدرجة: {avg:.1f}/10")
    print(f"نسبة الإخفاق: {len(failed)} من {len(results)}")

    return results
results = run_eval(dataset[:])